# deepFPlearn Prediction of AR, ER, and ED endpoints 

**Workflow:**
1. Setup environment and validate imports
2. Configure project paths
3. Load and prepare SMILES data
4. Run AR predictions (5-fold ensemble)
5. Run ER predictions (5-fold ensemble)
6. Run ED predictions (5-fold ensemble)
7. Ensemble results and export

In [ ]:
from pathlib import Path
import json

root = Path.cwd()
configs = sorted((root / "configs").glob("predict_*.json"))

print("Working directory:", root)
print("Number of config files:", len(configs))

for file in configs:
    with open(file, encoding="utf-8") as f:
        cfg = json.load(f)

    input_path = Path(cfg["test_path"])
    output_path = Path(cfg["preds_path"])
    model_path = Path(cfg["checkpoint_path"])

    print(f"\n{file.name}")
    print("  Input:", input_path)
    print("  Output:", output_path)
    print("  Model exists:", model_path.is_file())

    if input_path.is_absolute() or output_path.is_absolute():
        print("  WARNING: Absolute input/output path!")

## 1. Environment Setup & Validation

In [ ]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

# Import validation
required_packages = {
    'dfpl': 'deepfplearn',
    'rdkit': 'RDKit',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'sklearn': 'scikit-learn'
}

print("\n=== Checking Imports ===")
for package, name in required_packages.items():
    try:
        __import__(package)
        print(f"✓ {name}: OK")
    except ImportError as e:
        print(f"✗ {name}: MISSING")
        print(f"  Install: {sys.executable} -m pip install {name}")
        raise

## 2. Project Configuration

In [ ]:
from pathlib import Path
import os


# Set project root to the current working directory
PROJECT_ROOT = Path.cwd()

print(f"Project root: {PROJECT_ROOT}")
print(f"Current dir: {Path.cwd()}")

# Verify required directories exist
required_dirs = ['configs', 'data', 'results_predictgnn']
for dir_name in required_dirs:
    dir_path = PROJECT_ROOT / dir_name
    if dir_path.exists():
        print(f"✓ {dir_name}/ exists")
    else:
        print(f"⚠ {dir_name}/ not found - will be created")
        os.makedirs(dir_path, exist_ok=True)

## 3. Load & Prepare Data

In [ ]:
import pandas as pd

# Input Excel file
xlsx = "input.xlsx"
sheet_name = "general"

# Column names
id_col = "group"
score_col = "score.sirius"
smiles_col = "SMILES.sirius"

print(f"Loading {xlsx}...")
df = pd.read_excel(xlsx, sheet_name=sheet_name)
print(f"Loaded {len(df)} rows")

# Keep only needed columns
cols_present = [c for c in [id_col, score_col, smiles_col] if c in df.columns]
df = df[cols_present].copy()

# Clean data
df[score_col] = pd.to_numeric(df[score_col], errors="coerce")
df[smiles_col] = df[smiles_col].astype(str).str.strip()
df.replace({"": pd.NA}, inplace=True)

# Filter: both score and SMILES not NA
filtered = df.dropna(subset=[score_col, smiles_col])
filtered = filtered[filtered[smiles_col].str.lower() != "nan"]

# Rename for output
out = filtered.rename(columns={
    id_col: "id",
    score_col: "score",
    smiles_col: "smiles"
})

# Remove duplicates
out_ids = out.drop_duplicates(subset=["id"])
out_smiles = out[["smiles"]].drop_duplicates()

# Save
os.makedirs("data", exist_ok=True)
out_ids.to_csv("data/smiles_with_id.csv", index=False)
out_smiles.to_csv("data/smiles.csv", index=False)

print(f"✓ Wrote data/smiles_with_id.csv ({len(out_ids)} rows)")
print(f"✓ Wrote data/smiles.csv ({len(out_smiles)} unique SMILES)")

## 4. Run AR Predictions (5-Fold)

In [ ]:
import subprocess

endpoint = "AR"
folds = 5

print(f"Running {endpoint} predictions ({folds} folds)...\n")

for fold in range(folds):
    config = f"configs/predict_{endpoint}_fold{fold}.json"
    print(f"{endpoint} fold {fold}...", end=" ")

    result = subprocess.run(
        [sys.executable, "-m", "dfpl", "predictgnn", "-f", config],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("✓")
    else:
        print(f"✗ FAILED")
        print(f"  Error: {result.stderr}")
        raise RuntimeError(f"{endpoint} fold {fold} prediction failed")

print(f"\n✓ {endpoint} predictions complete")

## 5. Run ER Predictions (5-Fold)

In [ ]:
endpoint = "ER"
folds = 5

print(f"Running {endpoint} predictions ({folds} folds)...\n")

for fold in range(folds):
    config = f"configs/predict_{endpoint}_fold{fold}.json"
    print(f"{endpoint} fold {fold}...", end=" ")

    result = subprocess.run(
        [sys.executable, "-m", "dfpl", "predictgnn", "-f", config],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("✓")
    else:
        print(f"✗ FAILED")
        print(f"  Error: {result.stderr}")
        raise RuntimeError(f"{endpoint} fold {fold} prediction failed")

print(f"\n✓ {endpoint} predictions complete")

## 6. Run ED Predictions (5-Fold)

In [ ]:
endpoint = "ED"
folds = 5

print(f"Running {endpoint} predictions ({folds} folds)...\n")

for fold in range(folds):
    config = f"configs/predict_{endpoint}_fold{fold}.json"
    print(f"{endpoint} fold {fold}...", end=" ")

    result = subprocess.run(
        [sys.executable, "-m", "dfpl", "predictgnn", "-f", config],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("✓")
    else:
        print(f"✗ FAILED")
        print(f"  Error: {result.stderr}")
        raise RuntimeError(f"{endpoint} fold {fold} prediction failed")

print(f"\n✓ {endpoint} predictions complete")

## 7. Ensemble & Export Results

In [ ]:
import pandas as pd
import numpy as np
import glob

# Load original data with scores
excel_path = "input.xlsx"
sheet_general = "general"
id_col = "group"
score_col = "score.sirius"
smiles_col = "SMILES.sirius"

g = pd.read_excel(excel_path, sheet_name=sheet_general,
                  usecols=lambda c: c in {id_col, score_col, smiles_col})

# Clean
g[score_col] = pd.to_numeric(g[score_col], errors="coerce")
g[smiles_col] = g[smiles_col].astype(str).str.strip()
g = g.replace({"": pd.NA})
g = g.dropna(subset=[smiles_col])
g = (g.sort_values(score_col, ascending=False)
       .drop_duplicates(subset=[id_col]))

mapping_full = g.rename(columns={
    id_col: "id",
    score_col: "score",
    smiles_col: "smiles"
})

mapping_id_smiles = mapping_full[["id", "smiles"]].drop_duplicates()

# Setup directories
endpoints = ["AR", "ER", "ED"]
raw_dir = "results_predictgnn/raw"
out_dir = "results_predictgnn/ensemble"
os.makedirs(out_dir, exist_ok=True)

# Ensemble function
def ensemble_one(ep):
    files = sorted(glob.glob(f"{raw_dir}/{ep}_fold*.csv"))
    print(f"Ensembling {ep}: found {len(files)} fold files")

    if len(files) != 5:
        raise RuntimeError(f"{ep}: Expected 5 fold files, found {len(files)}")

    dfs = []
    for i, f in enumerate(files):
        df = pd.read_csv(f)
        # Find prediction column
        if ep in df.columns:
            col = ep
        else:
            cols = [c for c in df.columns if c.lower() != "smiles"]
            col = cols[-1] if cols else None

        if col is None:
            raise RuntimeError(f"No prediction column found in {f}")

        dfs.append(df[["smiles", col]].rename(columns={col: f"{ep}_fold{i}"}))

    # Merge all folds
    merged = dfs[0]
    for d in dfs[1:]:
        merged = merged.merge(d, on="smiles", how="inner")

    # Calculate mean and std
    fold_cols = [c for c in merged.columns if c.startswith(f"{ep}_fold")]
    merged[f"{ep}_mean"] = merged[fold_cols].mean(axis=1)
    merged[f"{ep}_std"] = merged[fold_cols].std(axis=1)

    # Save ensemble
    out_ep = f"{out_dir}/{ep}_ensemble.csv"
    merged.to_csv(out_ep, index=False)
    print(f"  ✓ Wrote {out_ep} ({len(merged)} rows)")

    # Attach IDs
    ens_id = merged.merge(mapping_id_smiles, on="smiles", how="left")
    ens_id = ens_id[["id"] + [c for c in ens_id.columns if c != "id"]]
    out_ep_id = f"{out_dir}/{ep}_ensemble_with_id.csv"
    ens_id.to_csv(out_ep_id, index=False)
    print(f"  ✓ Wrote {out_ep_id}")

    return ens_id

# Run ensemble for each endpoint
print("\n=== Ensemble Results ===")
for ep in endpoints:
    ensemble_one(ep)

print("\n✓ All ensembles complete")

## 8. Combine All Endpoints

In [ ]:
# Combine all three endpoints
combined = None
for ep in endpoints:
    t = pd.read_csv(f"{out_dir}/{ep}_ensemble_with_id.csv")[["id", "smiles", f"{ep}_mean", f"{ep}_std"]]
    combined = t if combined is None else combined.merge(t, on=["id", "smiles"], how="outer")

# Merge score.sirius by ID
combined = combined.merge(mapping_full[["id", "score"]], on="id", how="left")

# Reorder columns: id, score, smiles, then endpoint stats
base_cols = ["id", "score", "smiles"]
other_cols = [c for c in combined.columns if c not in base_cols]
combined = combined[base_cols + other_cols]

# Export final results
out_all = f"{out_dir}/ALL_endpoints_ensemble_with_id.csv"
combined.to_csv(out_all, index=False)

print(f"✓ Final results: {out_all}")
print(f"  Total rows: {len(combined)}")
print(f"  Columns: {list(combined.columns)}")
print(f"\n=== WORKFLOW COMPLETE ===")